# YOLO Training for Capsule Vision Inspection

This notebook trains a YOLOv8 nano detection model on the capsule dataset.

## Pipeline:
1. Install dependencies
2. Download/upload dataset
3. Verify dataset structure
4. Train YOLO
5. Validate
6. Download best.pt

In [ ]:
# Install dependencies
!pip install ultralytics

import os
from pathlib import Path
import yaml

from ultralytics import YOLO

print("Ultralytics version:", YOLO.__module__)

In [ ]:
# Configuration
DATASET_PATH = Path("/content/dataset")
DATA_YAML = DATASET_PATH / "data.yaml"

CLASS_NAMES = [
    "Good",
    "Crack",
    "Scratch",
    "Faulty Imprint",
    "Poke",
    "Squeeze",
    "Contamination",
]

# Training hyperparameters
EPOCHS = 50
IMGSZ = 640
BATCH = 16  # Adjust based on GPU memory
MODEL = "yolov8n.pt"  # YOLO nano detection model

print(f"Dataset: {DATASET_PATH}")
print(f"Model: {MODEL}")
print(f"Epochs: {EPOCHS}")
print(f"Image size: {IMGSZ}")
print(f"Batch: {BATCH}")

In [ ]:
# Verify dataset structure
for split in ["train", "val", "test"]:
    img_dir = DATASET_PATH / "images" / split
    lbl_dir = DATASET_PATH / "labels" / split
    n_imgs = len(list(img_dir.glob("*.*"))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{split}: {n_imgs} images, {n_lbls} labels")

# Check data.yaml
if DATA_YAML.exists():
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print("\ndata.yaml:")
    print(yaml.dump(data_cfg, default_flow_style=False))
else:
    print("\n⚠️  data.yaml not found! Creating it...")
    data_cfg = {
        "path": str(DATASET_PATH),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    }
    with open(DATA_YAML, "w") as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    print("Created data.yaml")

In [ ]:
# Train YOLO model
model = YOLO(MODEL)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,  # Early stopping
    project="runs/train",
    name="capsule_yolov8n",
    exist_ok=True,
    verbose=True,
)

print("\nTraining complete!")

In [ ]:
# Validate the best model
best_model = YOLO("runs/train/capsule_yolov8n/weights/best.pt")

metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    verbose=True,
)

print("\n=== Validation Results ===")
print(f"Precision: {metrics.box.p:.4f}")
print(f"Recall: {metrics.box.r:.4f}")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
# Per-class metrics
print("=== Per-Class Metrics ===")
print(f"{'Class':<20} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)

for i, name in enumerate(CLASS_NAMES):
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    ap = metrics.box.ap[i] if i < len(metrics.box.ap) else 0
    print(f"{name:<20} {p:>10.4f} {r:>10.4f} {ap50:>10.4f} {ap:>10.4f}")

In [ ]:
# Download best.pt for deployment
from google.colab import files

best_path = "runs/train/capsule_yolov8n/weights/best.pt"
print(f"Model saved at: {best_path}")
print(f"Model size: {os.path.getsize(best_path) / 1024 / 1024:.2f} MB")

# Download to local machine
files.download(best_path)

In [ ]:
# Optional: Test inference on a sample image
import cv2
import matplotlib.pyplot as plt

# Find a test image
test_dir = DATASET_PATH / "images" / "test"
test_images = list(test_dir.glob("*.*"))

if test_images:
    sample = str(test_images[0])
    results = best_model.predict(source=sample, conf=0.25, verbose=False)

    # Display result
    result_img = results[0].plot()
    plt.figure(figsize=(10, 8))
    plt.imshow(result_img)
    plt.axis("off")
    plt.title(f"Sample: {Path(sample).name}")
    plt.show()

    # Print detections
    for det in results[0].boxes:
        cls_id = int(det.cls)
        conf = float(det.conf)
        bbox = [int(v) for v in det.xyxy[0]]
        print(f"{CLASS_NAMES[cls_id]}: {conf:.2%} bbox={bbox}")
else:
    print("No test images found.")